# prototype clock value in jax
Make a working clock-value code with jax and jax-finufft that is fast enough!

## author:
- **David W. Hogg** (NYU) (Flatiron) (MPIA)

## bugs:
- Needs to be rearranged: Should do L-S periodogram first and then use those as hypotheses.

## comments:
- Because my Mac has issues with jax, only use jax when we absolutely need it.
- Takes *way* too long. Ought to do a first pass with L-S and only run clock value on three-point regions centered on L-S peaks.

In [ ]:
# !pip install lightkurve
# !pip install jax

In [ ]:
import time
from functools import partial
import numpy as np
import jax
import jax.numpy as jnp
import lightkurve as lk
import matplotlib.pyplot as plt

In [ ]:
jax.config.update('jax_platform_name', 'cpu') # because bad Mac behavior?
jax.config.update("jax_enable_x64", True)

In [ ]:
def get_kepler_data(kic_id, exptime='long'):
    """
    ## Inputs:
    `kic_id`: Kepler ID (str)
    `exptime`: default exposure time, 'long'

    ## Outputs:
    Returns a 5-tuple:
    - `times`, `fluxes`, `errors`: light curve data
    - `delta_f`: frequency resolution, 1 / total observation time
    - `sampling_time`: median time between observations (in days)
    """    
    start = time.time()
    print("starting to obtain data for", kic_id)

    try:
        search_result = lk.search_lightcurve(kic_id, mission = 'Kepler', exptime=exptime)
        if len(search_result) < 1:
            msg = f"get_kepler_data(): no results for {kic_id} at this cadence"
            print(msg)
            update_error_message(kic_id, 'Kepler_long', msg)
            return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lk.search_lightcurve() failed for {kic_id } with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        lc_collection = search_result.download_all()
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): search_result.download_all() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    try:
        lc = lc_collection.stitch()
        #print("get_kepler_data(): minimum time value", np.min(lc.time.value), np.min(lc.time), lc.time)
    except lk.LightkurveError as e:
        print(f"LightkurveError for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan
    except Exception as e:
        print(f"Exception for {kic_id}: get_kepler_data(): lc_collection.stitch() failed for {kic_id} with {str(e)}")
        return np.nan, np.nan, np.nan, np.nan, np.nan

    # unpack, remove bad data, and reorder
    times, fluxes, errors = lc.time.value, lc.flux.value, lc.flux_err.value
    good = np.isfinite(times) & np.isfinite(fluxes) & np.isfinite(errors)
    times, fluxes, errors = times[good], fluxes[good], errors[good]
    idx = np.argsort(times)
    times, fluxes, errors = times[idx], fluxes[idx], errors[idx]

    delta_f = (1/(times[-1] - times[0]))
    sampling_time= np.median(np.diff(times))
    print("get_kepler_data() took", time.time() - start, "seconds")

    return times, fluxes, errors, delta_f, sampling_time

In [ ]:
# get data on a good example

ts, ys, errs, deltaf, deltat = get_kepler_data("KIC005285607")
print(ts.shape, ys.shape, errs.shape, deltaf, deltat)

In [ ]:
# check the data

f = plt.figure(figsize=(9, 3))
plt.scatter(ts, ys, s=1, c="k", marker=".")

In [ ]:
# get a set of hypotheses for this star

def get_candidate_frequencies(times, fluxes, errors, delta_f, delta_t):

In [ ]:
# start by solving this problem with standard (non-finufft) methods

@partial(jax.jit, static_argnums=2)
def design_matrix(om, t, M):
    ms1, ms2 = jnp.arange(M), jnp.arange(1, M)
    return jnp.concat((jnp.cos(ms1[None, :] * om * t[:, None]),
                      jnp.sin(ms2[None, :] * om * t[:, None])), axis=1), \
            jnp.concat((ms1, ms2))                    

In [ ]:
# time to get the clock value

@partial(jax.jit, static_argnums=4)
def clock_value(om, t, y, iv, M):
    """
    # bugs:
    - doesn't use jax_finufft
    - This is very affected by outliers; need to remove those somehow. But *don't use `jnp.median()`!
    - maybe should use IRLS to do the fit
    """
    X, m = design_matrix(om, t, M)
    A = X.T @ (iv[:, None] * X)
    b = X.T @ (iv * ys)
    pars = jnp.linalg.solve(A, b)
    mse = jnp.mean((y - X @ pars) ** 2)
    return (om ** 2 / mse) * jnp.sum(m ** 2 * pars ** 2)

clock_values = jax.vmap(clock_value, in_axes=(0, None, None, None, None))

In [ ]:
# now do all frequencies
# BUG: NOT CURRENTLY DOING ALL FREQUENCIES BECAUSE HARDWARE SUX

start = time.time()

ivars = 1. / errs ** 2
smallest = 2. * jnp.pi / 3. # smallest angular frequency to consider MAGIC 3-day period
nyquist = jnp.pi / deltat
M = 5 # magic

best_value = 0.
best_omega = 0.
oms = jnp.arange(smallest, smallest + 0.01 * nyquist, 0.25 * jnp.pi * deltaf) # oversampling by a factor of 8-ish MAGIC
values = clock_values(oms, ts, ys, ivars, M)

print("cell took", time.time() - start, "seconds")
print(oms.shape, values.shape)

In [ ]:
# get best frequency
# Note: this is the parabola trick in the log

@jax.jit
def get_best_peak(xs, ys):
    ii = jnp.argmax(ys)
    foo = jnp.polyfit(jax.lax.dynamic_slice(xs, (ii-1, ), (3, )),
                      jax.lax.dynamic_slice(jnp.log(ys), (ii-1, ), (3, )), 2)
    root = jnp.roots(jnp.polyder(foo), strip_zeros=False).real
    return root,jnp.exp(jnp.polyval(foo, root))

best_omega, best_value = get_best_peak(oms, values)
print(best_omega, best_value)

In [ ]:
plt.plot(oms, values, "k.")
plt.plot(oms, values, "k-")
plt.scatter(best_omega, best_value, marker="o", c="r")
# plt.xlim(1.6, 1.62)

In [ ]:
# look at best frequency

best_T = 2. * jnp.pi / best_omega
f = plt.figure(figsize=(9, 3))
plt.scatter(ts % best_T, ys, s=1, c="k", marker=".")